# Summarization benchmark with MLflow

Benchmark the same Walden chapters across models. Each model configuration and repetition creates one MLflow run containing aggregate metrics, per-call measurements, prompts, and generated summaries.

Start the existing Docker service from the project directory:

```bash
uv sync
docker compose up -d mlflow
```

Open http://127.0.0.1:5000 and select the project Jupyter kernel. This notebook runs on the host and sends tracking data to Docker. If Jupyter also runs in Compose, use `http://mlflow:5000` instead.

Run the configuration and preparation cells first. The benchmark cell makes real model requests and can incur API charges. Nothing starts in the background.


In [1]:
from pathlib import Path

import hashlib
import itertools
import json
import math
import os
import platform
import re
import statistics
import traceback
from collections.abc import Callable, Iterator
from time import perf_counter
from uuid import uuid4

import mlflow
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_openrouter import ChatOpenRouter
from tqdm.auto import tqdm
from transformers import AutoTokenizer

# Resolve project paths whether Jupyter starts in the root or notebook folder.
PROJECT_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project root or one of its subfolders.")

load_dotenv(PROJECT_ROOT / ".env")


/home/andakaraas/workspace/llm-summarization/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


True

## Model configurations

The initial entry uses the existing `.env` provider/model. Add entries to `MODEL_CONFIGS` to benchmark more models. Use the matching `tokenizer` for each model, especially when provider usage is unavailable. Estimates are labeled separately from provider-reported counts.

Use the same prompts, chapter split, temperature, and output limit where supported. Record unavoidable differences as separate configurations. For Ollama, pull each model before running; `num_gpu=0` preserves the existing CPU-only setup.

`REPEATS=3` runs the complete book three times per configuration. For a quick check, set `SUMMARIZATION_TESTING=true` in `.env` to use five chapters and temporarily set `REPEATS=1`. Run the separate sanity check cell once per configuration before starting the experiment. Sanity requests are excluded from benchmark timing and token counts. Cache state and hosted-provider load can still vary between repetitions.


In [2]:
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "summarization-latency-benchmark")
REPEATS = 1
TESTING = os.getenv("SUMMARIZATION_TESTING", "false").lower() == "true"

provider = os.getenv("PROVIDER", "ollama").strip().lower()
MODEL_CONFIGS = [{
    "provider": provider,
    "model": (
        os.getenv("OLLAMA_MODEL", "LiquidAI/lfm2.5-2.6b:q4_k_m")
        if provider == "ollama" else os.getenv("OPENROUTER_MODEL", "")
    ),
    "tokenizer": os.getenv("TOKENIZER_NAME", "LiquidAI/LFM2.5-2.6B"),
    "temperature": 0.1,
    "max_output_tokens": int(os.getenv("MAX_OUTPUT_TOKENS", "2048")),
    # Used only for Ollama:
    "num_ctx": int(os.getenv("OLLAMA_NUM_CTX", "32768")),
    "num_gpu": 0,
}]
# Example: append another OpenRouter configuration with its matching tokenizer.
# MODEL_CONFIGS.append({
#     "provider": "openrouter", "model": "provider/model-name",
#     "tokenizer": "organization/tokenizer-name", "temperature": 0.1,
#     "max_output_tokens": 2048,
# })

SYSTEM_PROMPT = "You are a careful document summarizer. Use only the supplied source documents."
CHAPTER_PROMPT = (
    "Using only the book chapter document, compose a summary of the book chapter.\n"
    "Your response should only include the summary. Do not provide any further explanation."
)
FINAL_PROMPT = (
    "Using only the book chapter summary documents, compose a single, unified summary of the book.\n"
    "Your response should only include the unified summary. Do not provide any further explanation."
)
SOURCE_URL = "https://www.gutenberg.org/cache/epub/205/pg205-images.html"

if REPEATS < 1 or not MODEL_CONFIGS:
    raise ValueError("Set REPEATS >= 1 and provide at least one model configuration.")
for config in MODEL_CONFIGS:
    if config["provider"] not in {"ollama", "openrouter"} or not config["model"].strip():
        raise ValueError("Each configuration needs a valid provider and nonempty model name.")
    if config["max_output_tokens"] <= 0:
        raise ValueError("max_output_tokens must be positive.")
    if config["provider"] == "openrouter" and not os.getenv("OPENROUTER_API_KEY"):
        raise ValueError("OPENROUTER_API_KEY is missing from .env.")

mlflow.set_tracking_uri(TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow: {TRACKING_URI}; experiment: {EXPERIMENT_NAME}")
print(f"Planned runs: {len(MODEL_CONFIGS) * REPEATS}; first five chapters only: {TESTING}")
display(pd.DataFrame(MODEL_CONFIGS))


MLflow: http://127.0.0.1:5000; experiment: summarization-latency-benchmark
Planned runs: 1; first five chapters only: False


,provider,model,tokenizer,temperature,max_output_tokens,num_ctx,num_gpu
0,ollama,LiquidAI/lfm2.5-2.6b:q4_k_m,LiquidAI/LFM2.5-2.6B,0.1,2048,65536,0


## Prepare the document once

Reuse the original Docling chapter extraction for every model and repetition. Download, parsing, chunking, tokenizer/model initialization, sanity checks, and MLflow uploads are outside `summarization_seconds`. The dataset fingerprint includes chapter order, IDs, titles, and text.


In [3]:
import itertools
from collections.abc import Callable, Iterator

from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker


def chunk_document(source: str, *, dropwhile: Callable[[BaseChunk], bool] = lambda c: False, takewhile: Callable[[BaseChunk], bool] = lambda c: True) -> Iterator[BaseChunk]:
    """Read the document and perform a hierarchical chunking"""
    converter = DocumentConverter()
    chunks = HierarchicalChunker().chunk(converter.convert(source=source).document)
    return itertools.takewhile(takewhile, itertools.dropwhile(dropwhile, chunks))

def merge_chunks(chunks: Iterator[BaseChunk], *, headings: Callable[[BaseChunk], list[str]] = lambda c: c.meta.headings) -> Iterator[dict[str, str]]:
    """Merge chunks having the same headings"""
    prior_headings: list[str] | None = None
    document: dict[str, str] = {}
    doc_id = 0
    for chunk in chunks:
        text = chunk.text.replace('\r\n', '\n')
        current_headings = headings(chunk)
        if prior_headings != current_headings:
            if document:
                yield document
            prior_headings = current_headings
            document = {
                'doc_id': str(doc_id := doc_id + 1),
                'title': " - ".join(current_headings),
                'text': text
            }
        else:
            document['text'] += f"\n\n{text}"
    if document:
        yield document

def chunk_dropwhile(chunk: BaseChunk) -> bool:
    """Ignore front matter prior to the book start"""
    return chunk.meta.headings != ["WALDEN", "Economy"]

def chunk_takewhile(chunk: BaseChunk) -> bool:
    """Ignore remaining chunks once we see this heading"""
    return "ON THE DUTY OF CIVIL DISOBEDIENCE" not in chunk.meta.headings

def chunk_headings(chunk: BaseChunk) -> list[str]:
    """Use the h1 and h2 (chapter) headings"""
    return chunk.meta.headings[:2]


documents = list(merge_chunks(
    chunk_document(SOURCE_URL, dropwhile=chunk_dropwhile, takewhile=chunk_takewhile),
    headings=chunk_headings,
))
if TESTING:
    documents = documents[:5]
if not documents or any(not document["text"].strip() for document in documents):
    raise ValueError("Expected nonempty Walden chapters. Check the source and heading filters.")

source_text = "\n\n".join(document["text"] for document in documents)
source_characters = len(source_text)
source_words = len(source_text.split())
dataset_sha256 = hashlib.sha256(
    json.dumps(documents, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()
prompt_sha256 = hashlib.sha256(
    json.dumps([SYSTEM_PROMPT, CHAPTER_PROMPT, FINAL_PROMPT]).encode("utf-8")
).hexdigest()
print(f"{len(documents)} chapters; {source_words:,} words; {source_characters:,} characters")
print(f"Dataset SHA256: {dataset_sha256}")


18 chapters; 106,505 words; 581,844 characters
Dataset SHA256: dddef76265c768ce5131bc9aa3dbb05beb98af463f1670e4c8b6cf16984e5feb


## Model calls and timing

Calls are sequential with automatic client retries disabled. Each row records stage, chapter, latency, token counts, count provenance, and finish reason. Output limits are not target summary lengths; inspect artifacts for truncation and quality.

For models without provider usage, input counts use the selected tokenizer's chat template when available, otherwise plain message text. These estimates may differ from the provider's formatting. Source token counts also depend on the selected tokenizer, so prefer `source_words_per_second` when comparing document processing speed across models.


In [4]:
THINKING_BLOCK_PATTERN = re.compile(r"<think>.*?</think>", re.IGNORECASE | re.DOTALL)
DOCUMENT_TAG_PATTERN = re.compile(r"</?document(?:\s+[^>]*)?>", re.IGNORECASE)


def clean_summary(text):
    return DOCUMENT_TAG_PATTERN.sub("", THINKING_BLOCK_PATTERN.sub("", text)).strip()


def create_model(config):
    common = {"model": config["model"], "temperature": config["temperature"], "max_retries": 0}
    if config["provider"] == "ollama":
        # Ollama does not use the OpenRouter retry option.
        common.pop("max_retries")
        return ChatOllama(
            **common,
            base_url=os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/"),
            validate_model_on_init=True,
            reasoning=False,
            num_ctx=config["num_ctx"],
            num_gpu=config["num_gpu"],
            num_predict=config["max_output_tokens"],
            top_k=50,
            repeat_penalty=1.1,
        )
    return ChatOpenRouter(**common, max_tokens=config["max_output_tokens"])


def generate(model, tokenizer, config, user_prompt, call_documents):
    document_text = "\n\n".join(
        f"<document id='{doc['doc_id']}' title='{doc['title']}'>\n{doc['text']}\n</document>"
        for doc in call_documents
    )
    request_text = f"{user_prompt}\n\nSource documents:\n{document_text}"
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request_text},
    ]
    if tokenizer.chat_template:
        estimated_input_tokens = len(tokenizer.apply_chat_template(
            chat, tokenize=True, add_generation_prompt=True,
        ))
    else:
        estimated_input_tokens = len(tokenizer.encode(
            SYSTEM_PROMPT + "\n\n" + request_text, add_special_tokens=False,
        ))
    if config["provider"] == "ollama":
        if estimated_input_tokens + config["max_output_tokens"] > config["num_ctx"]:
            raise ValueError("Input plus output budget exceeds num_ctx; use smaller chapters or a larger context.")

    started_at = perf_counter()
    response = model.invoke([
        SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=request_text),
    ])
    elapsed_seconds = perf_counter() - started_at
    content = response.content
    text = content if isinstance(content, str) else "".join(
        block if isinstance(block, str) else block.get("text", "")
        for block in content
    )
    if not clean_summary(text):
        raise ValueError("The model returned an empty final summary.")
    usage = response.usage_metadata or {}
    input_tokens = usage.get("input_tokens")
    output_tokens = usage.get("output_tokens")
    count_source = "provider" if input_tokens is not None and output_tokens is not None else "mixed_or_estimated"
    if input_tokens is None:
        input_tokens = estimated_input_tokens
    if output_tokens is None:
        output_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    metadata = response.response_metadata or {}
    return text, {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "elapsed_seconds": elapsed_seconds,
        "token_count_source": count_source,
        "finish_reason": metadata.get("finish_reason") or metadata.get("done_reason"),
    }


In [5]:
def run_summarization(model, tokenizer, config, call_metrics, summaries):
    # Mutable buffers preserve completed calls if a later request fails.
    with tqdm(total=len(documents) + 1, desc="Summarization", unit="call", dynamic_ncols=True) as progress:
        try:
            for index, document in enumerate(documents, start=1):
                progress.set_description(f"Chapter {index}/{len(documents)}: {document['title']}")
                output, metrics = generate(model, tokenizer, config, CHAPTER_PROMPT, [document])
                call_metrics.append({"stage": "chapter", "title": document["title"], **metrics})
                summaries.append({
                    "doc_id": document["doc_id"],
                    "title": document["title"],
                    "text": clean_summary(output),
                })
                progress.set_postfix(last_call=f"{metrics['elapsed_seconds']:.1f}s", refresh=False)
                progress.update(1)

            progress.set_description("Final synthesis")
            output, metrics = generate(model, tokenizer, config, FINAL_PROMPT, summaries)
            call_metrics.append({"stage": "final_synthesis", "title": "Walden", **metrics})
            progress.set_postfix(last_call=f"{metrics['elapsed_seconds']:.1f}s", refresh=False)
            progress.update(1)
            progress.set_description("Summarization complete")
            return clean_summary(output)
        except BaseException as error:
            stage = "Interrupted" if isinstance(error, KeyboardInterrupt) else "Failed"
            progress.set_description(f"{stage}: {progress.desc}")
            raise


def benchmark_one(model, config, config_index, repeat_index, batch_id):
    with mlflow.start_run(
        run_name=f"{config['model']}-config-{config_index}-repeat-{repeat_index}",
        tags={"batch_id": batch_id, "timing_scope": "summarization_only"},
    ) as run:
        params = {
            key: value for key, value in config.items()
            if config["provider"] == "ollama" or key not in {"num_ctx", "num_gpu"}
        }
        mlflow.log_params({
            **params,
            "config_index": config_index,
            "repeat_index": repeat_index,
            "dataset_sha256": dataset_sha256,
            "prompt_sha256": prompt_sha256,
            "chapter_count": len(documents),
            "testing": TESTING,
            "concurrency": 1,
            "sanity_check": "before_experiment",
            "client_retries": 0,
        })
        if config["provider"] == "ollama":
            mlflow.log_params({"reasoning": False, "top_k": 50, "repeat_penalty": 1.1})
        mlflow.log_dict({
            "source_url": SOURCE_URL,
            "system": SYSTEM_PROMPT, "chapter": CHAPTER_PROMPT, "final": FINAL_PROMPT,
            "platform": platform.platform(), "python": platform.python_version(),
            "mlflow": mlflow.__version__,
            "ollama_host": os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434") if config["provider"] == "ollama" else None,
        }, "config/context.json")
        tokenizer = AutoTokenizer.from_pretrained(config["tokenizer"])
        source_tokens = len(tokenizer.encode(source_text, add_special_tokens=False))

        calls, summaries = [], []
        started_at = perf_counter()
        try:
            final_summary = run_summarization(model, tokenizer, config, calls, summaries)
        except Exception:
            elapsed = perf_counter() - started_at
            mlflow.log_metrics({"failed_calls": 1, "completed_calls": len(calls), "elapsed_before_failure_seconds": elapsed})
            mlflow.log_dict({"calls": calls}, "metrics/per_call.json")
            mlflow.log_dict({"chapters": summaries}, "outputs/partial_summaries.json")
            mlflow.log_text(traceback.format_exc(), "errors/traceback.txt")
            raise  # The run context records FAILED; the batch continues with the next run.
        elapsed = perf_counter() - started_at

        input_tokens = sum(item["input_tokens"] for item in calls)
        output_tokens = sum(item["output_tokens"] for item in calls)
        total_tokens = input_tokens + output_tokens
        latencies = sorted(item["elapsed_seconds"] for item in calls)
        metrics = {
            "source_characters": source_characters,
            "source_words": source_words,
            "source_tokens": source_tokens,
            "model_calls": len(calls),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_model_tokens": total_tokens,
            "model_call_seconds": sum(latencies),
            "summarization_seconds": elapsed,
            "total_tokens_per_second": total_tokens / elapsed,
            "output_tokens_per_second": output_tokens / elapsed,
            "source_words_per_second": source_words / elapsed,
            "pipeline_token_multiplier": total_tokens / source_tokens,
            "call_latency_median_seconds": statistics.median(latencies),
            "call_latency_p95_seconds": latencies[math.ceil(0.95 * len(latencies)) - 1],
            "truncated_calls": sum(item["finish_reason"] in {"length", "max_tokens"} for item in calls),
            "failed_calls": 0,
        }
        mlflow.set_tag("token_count_source", "provider" if all(
            item["token_count_source"] == "provider" for item in calls
        ) else "mixed_or_estimated")
        tqdm.write("Saving results to MLflow...")
        # Upload after stopping the timer so tracking traffic does not inflate throughput timing.
        mlflow.log_metrics(metrics)
        mlflow.log_dict({"calls": calls}, "metrics/per_call.json")
        mlflow.log_dict({"chapters": summaries}, "outputs/chapter_summaries.json")
        mlflow.log_text(final_summary, "outputs/final_summary.txt")
        return {"run_id": run.info.run_id, "model": config["model"], **metrics}


## Sanity check before the experiment

Run this cell once before the benchmark cell. It calls each configured model once and displays the response. An error or empty response stops this cell before any experiment run starts. Rerun this cell after changing `MODEL_CONFIGS`.


In [6]:
models = []
checked_configs = [dict(config) for config in MODEL_CONFIGS]

for config in MODEL_CONFIGS:
    print(f"Sanity check: {config['provider']}/{config['model']}", flush=True)
    model = create_model(config)
    response = model.invoke("Reply with OK.")
    content = response.content
    sanity_text = content if isinstance(content, str) else "".join(
        block if isinstance(block, str) else block.get("text", "")
        for block in content
    )
    if not sanity_text.strip():
        raise ValueError(f"Sanity check returned an empty response for {config['model']}.")
    print(f"Sanity check response: {sanity_text}", flush=True)
    models.append(model)


Sanity check: ollama/LiquidAI/lfm2.5-2.6b:q4_k_m
Sanity check response: <think>The user is asking me to reply with "OK". This is a simple, direct instruction. I should comply and respond exactly as requested.</think>OK


In [7]:
# from openrouter.errors import TooManyRequestsResponseError

# try:
#     response = model.invoke("Reply with OK.")
# except TooManyRequestsResponseError as error:
#     print("Error body:", error.body)
#     print("Retry-After:", error.raw_response.headers.get("retry-after"))
#     raise

## Run the benchmark

This cell executes all configurations sequentially, once per repetition. A fresh batch ID separates reruns of this cell. Failed runs remain FAILED in MLflow and do not enter the successful-run comparison. Interrupt the kernel to stop the batch.

Run the sanity check cell first. All repetitions reuse its model instances and do not send additional sanity requests. Sanity checks happen outside MLflow runs and warm the models before measurement. Provider-side caching and retries are outside this notebook's control.

The batch bar counts finished runs, including failures. The inner bar counts completed chapter calls plus final synthesis, shows the current stage and last-call latency, and closes on failure or interruption. Elapsed time and ETA refresh at request boundaries; ETA is approximate because chapter sizes vary. Results are uploaded after summarization timing stops.


In [ ]:
if "checked_configs" not in globals() or checked_configs != MODEL_CONFIGS or len(models) != len(MODEL_CONFIGS):
    raise RuntimeError("Run the sanity check cell for the current MODEL_CONFIGS first.")

batch_id = uuid4().hex
results = []
failures = []
print(f"Batch: {batch_id}")
with tqdm(total=REPEATS * len(MODEL_CONFIGS), desc="Batch", unit="run", dynamic_ncols=True) as batch_progress:
    for repeat_index in range(1, REPEATS + 1):
        for config_index, config in enumerate(MODEL_CONFIGS, start=1):
            run_number = (repeat_index - 1) * len(MODEL_CONFIGS) + config_index
            batch_progress.set_postfix(model=config["model"], repeat=f"{repeat_index}/{REPEATS}")
            tqdm.write(f"Run {run_number}/{batch_progress.total}: {config['provider']}/{config['model']}, repeat {repeat_index}/{REPEATS}")
            try:
                result = benchmark_one(models[config_index - 1], config, config_index, repeat_index, batch_id)
            except Exception as error:
                failures.append({"model": config["model"], "config_index": config_index, "repeat_index": repeat_index, "error": str(error)})
                tqdm.write(f"FAILED: {error}")
            else:
                results.append(result)
                tqdm.write(f"{result['summarization_seconds']:.2f} s; {result['total_tokens_per_second']:.2f} total tokens/s")
            batch_progress.set_postfix(successful=len(results), failed=len(failures), refresh=False)
            batch_progress.update(1)

print(f"Finished: {len(results)} successful, {len(failures)} failed. Batch: {batch_id}")
if failures:
    display(pd.DataFrame(failures))


Batch: 79be352eb2c14ad08c13691727f4e355


Batch:   0%|          | 0/1 [00:00<?, ?run/s]

Run 1/1: ollama/LiquidAI/lfm2.5-2.6b:q4_k_m, repeat 1/1


Summarization:   0%|          | 0/19 [00:00<?, ?call/s]

## Compare successful runs

The table is restricted to this batch. Compare median summarization time and its min/max across repetitions. `total_tokens_per_second` counts input plus output; `output_tokens_per_second` includes input processing and network time, so it is not isolated decoding speed. Throughput is computed from totals, never by averaging per-call rates.

Use source words/s to compare processing the same book across different tokenizers. Inspect finish reasons, truncation counts, and summary artifacts before choosing a faster model. There is no automatic quality score in this benchmark. MLflow's own Duration includes setup and uploads; use the explicit `summarization_seconds` metric instead.


In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.batch_id = '{batch_id}' AND attributes.status = 'FINISHED'",
)
if runs.empty:
    print("No successful runs in this batch. Inspect failures in MLflow.")
else:
    comparison = runs.groupby(["params.provider", "params.model", "params.config_index"]).agg(
        successful_runs=("run_id", "count"),
        median_seconds=("metrics.summarization_seconds", "median"),
        min_seconds=("metrics.summarization_seconds", "min"),
        max_seconds=("metrics.summarization_seconds", "max"),
        median_total_tokens_per_second=("metrics.total_tokens_per_second", "median"),
        median_output_tokens_per_second=("metrics.output_tokens_per_second", "median"),
        median_source_words_per_second=("metrics.source_words_per_second", "median"),
        truncated_calls=("metrics.truncated_calls", "sum"),
    ).sort_values("median_seconds")
    display(comparison)
    display(runs[[
        "run_id", "params.model", "params.repeat_index", "tags.token_count_source",
        "metrics.summarization_seconds", "metrics.input_tokens", "metrics.output_tokens",
    ]])


,,,successful_runs,median_seconds,min_seconds,max_seconds,median_total_tokens_per_second,median_output_tokens_per_second,median_source_words_per_second,truncated_calls
params.provider,params.model,params.config_index,,,,,,,,
openrouter,nvidia/nemotron-3-ultra-550b-a55b:free,1,1,250.814768,250.814768,250.814768,606.296037,46.177504,424.63608,0.0


,run_id,params.model,params.repeat_index,tags.token_count_source,metrics.summarization_seconds,metrics.input_tokens,metrics.output_tokens
0,136ff34358274138ac9a147ba36318e2,nvidia/nemotron-3-ultra-550b-a55b:free,1,provider,250.814768,140486.0,11582.0
